# Clamped and face-clamped edges

A clamped edge of the third-order shear deformation theory (TSDT) removes
the slope of $w$ normal to the edge together with the rotations, `x1wr =
0` and `x1phix = x1phiy = 0` on $x = 0$, such that the cross-section
stays planar and unrotated; an edge clamped by its faces, $u = v = 0$ at
$z = d \mp h/2$ and $w = 0$, leaves the section free to warp and is
prescribed with the keyword `z` of `Shell.add_distr_pd_fixed_x`,
see [Boundary conditions](../boundary_conditions.rst). The square plates below, $h/a = 0.1$, are
clamped at $x = 0, a$, free at $y = 0, b$, and loaded by a uniform
pressure:

In [1]:
import numpy as np
from structsolve import solve
from structsolve.sparseutils import finalize_symmetric_matrix
from panels.shell import Shell
from panels.multidomain import MultiDomain
from panels.multidomain.connections import nullspace, fkCld_xcte, fkCpd
from panels.multidomain.connections import kCpd

In [2]:
# definitions of tests/multidomain/test_face_clamp.py used below
A = 0.5
B = 0.5
E = 70e9
FIELDS = ('u', 'v', 'w', 'phix', 'phiy')
H = 0.05
NU = 0.3
Q = 1e6

In [3]:
# from tests/multidomain/test_face_clamp.py
def plate(model, offset=0., m=10, n=10):
    s = Shell(model=model, a=A, b=B, stack=[0.], plyt=H, laminaprop=(E, NU),
              offset=offset, m=m, n=n)
    for edge in ('x1', 'x2', 'y1', 'y2'):
        for field in FIELDS:
            setattr(s, edge + field, 1.)
            setattr(s, edge + field + 'r', 1.)
    s.add_pressure_load(Q)
    return s

In [4]:
# from tests/multidomain/test_face_clamp.py
def clamp_by_flags(s, edge):
    for field in FIELDS:
        setattr(s, edge + field, 0.)
    if 'fsdt' not in s.model:
        setattr(s, edge + 'wr', 0.)

In [5]:
# from tests/multidomain/test_face_clamp.py
def face_clamp(s, x, h, k=1., cte=True):
    zero = lambda y: 0.
    for z in (s.offset - h/2, s.offset + h/2):
        s.add_distr_pd_fixed_x(x, ku=k, kv=k, funcu=zero, funcv=zero, z=z,
                               cte=cte)
    s.add_distr_pd_fixed_x(x, kw=k, funcw=zero, cte=cte)

For the CLPT and the FSDT the face clamp is the clamp of the flags:

In [6]:
# definitions of tests/multidomain/test_face_clamp.py used below
def fields(s, c, xs, ys):
    xs = np.atleast_1d(np.asarray(xs, dtype=float))
    ys = np.atleast_1d(np.asarray(ys, dtype=float))
    c = np.ascontiguousarray(c)
    s.uvw(c, xs=xs, ys=ys)
    out = {k: s.fields[k].ravel().copy() for k in FIELDS}
    if 'clpt' in s.model:
        out['gxz'] = out['gyz'] = 0*out['u']
    else:
        s.strain(c, xs=xs, ys=ys, NLgeom=False)
        out['gxz'] = s.fields['gxz'].ravel().copy()
        out['gyz'] = s.fields['gyz'].ravel().copy()
    return out


def static(s, pd_method='null-space'):
    s._rebuild()
    md = MultiDomain([s], conn=[], pd_method=pd_method)
    return md, md.linear_static()

In [7]:
# from tests/multidomain/test_face_clamp.py
def test_face_clamp_equals_clamp_for_clpt_and_fsdt(model):
    s = plate(model)
    for x in (0., A):
        face_clamp(s, x, H)
    _, c = static(s)
    t = plate(model)
    clamp_by_flags(t, 'x1')
    clamp_by_flags(t, 'x2')
    _, ct = static(t)
    xs, ys = np.meshgrid(np.linspace(0, A, 7), np.linspace(0, B, 5))
    f = fields(s, c, xs.ravel(), ys.ravel())
    ft = fields(t, ct, xs.ravel(), ys.ravel())
    for k in FIELDS:
        assert np.allclose(f[k], ft[k], rtol=0, atol=1e-8*np.abs(ft['w']).max())

In [8]:
for model in ['plate_clpt_donnell', 'plate_fsdt_donnell']:
    test_face_clamp_equals_clamp_for_clpt_and_fsdt(model=model)
    print(f"test_face_clamp_equals_clamp_for_clpt_and_fsdt(model={model!r}): passed")

test_face_clamp_equals_clamp_for_clpt_and_fsdt(model='plate_clpt_donnell'): passed
test_face_clamp_equals_clamp_for_clpt_and_fsdt(model='plate_fsdt_donnell'): passed


For the TSDT, $u(z) = 0$ prescribed at a third coordinate gives the planar
clamp of the flags, which the former clamp with the slope of $w$ free does
not:

In [9]:
# definitions of tests/multidomain/test_face_clamp.py used below
def center_w(s, c):
    return fields(s, c, A/2, B/2)['w'][0]

In [10]:
# from tests/multidomain/test_face_clamp.py
def test_planar_clamp_is_u_zero_at_three_coordinates(offset):
    model = 'plate_tsdt_donnell'
    s = plate(model, offset)
    zero = lambda y: 0.
    for x in (0., A):
        face_clamp(s, x, H)
        s.add_distr_pd_fixed_x(x, ku=1., kv=1., funcu=zero, funcv=zero,
                               z=offset + H/4)
    _, c = static(s)
    t = plate(model, offset)
    clamp_by_flags(t, 'x1')
    clamp_by_flags(t, 'x2')
    _, ct = static(t)
    xs, ys = np.meshgrid(np.linspace(0, A, 7), np.linspace(0, B, 5))
    f = fields(s, c, xs.ravel(), ys.ravel())
    ft = fields(t, ct, xs.ravel(), ys.ravel())
    for k in FIELDS:
        assert np.allclose(f[k], ft[k], rtol=0, atol=1e-8*np.abs(ft['w']).max())
    # the former clamp of the shear deformation theories, with w,x free
    t.x1wr = t.x2wr = 1.
    _, cf = static(t)
    assert center_w(t, cf) > 1.05*center_w(t, ct)

In [11]:
for offset in [0.0, 0.3 * H]:
    test_planar_clamp_is_u_zero_at_three_coordinates(offset=offset)
    print(f"test_planar_clamp_is_u_zero_at_three_coordinates(offset={offset!r}): passed")

test_planar_clamp_is_u_zero_at_three_coordinates(offset=0.0): passed
test_planar_clamp_is_u_zero_at_three_coordinates(offset=0.015): passed


The face clamp lies between the planar clamp and the simple support, here
with the central deflections $w D/(q a^4)$ = 0.002901 (planar clamp),
0.002953 (face clamp) and 0.013459 (simple support): the displacements of
both faces vanish along the edges, whereas the section warps, with
$\phi_x = w_{,x}/2$:

In [12]:
# definitions of tests/multidomain/test_face_clamp.py used below
def uv_at(s, f, z, h):
    c1 = 4/(3*h**2) if 'tsdt' in s.model else 0.
    d = s.offset
    p = -c1*((z - d)**3 + d**3)
    return (f['u'] + z*f['phix'] + p*f['gxz'],
            f['v'] + z*f['phiy'] + p*f['gyz'])

In [13]:
# from tests/multidomain/test_face_clamp.py
def test_face_clamp_tsdt_thick_plate():
    model = 'plate_tsdt_donnell'
    s = plate(model)
    for x in (0., A):
        face_clamp(s, x, H)
    _, c = static(s)
    planar = plate(model)
    clamp_by_flags(planar, 'x1')
    clamp_by_flags(planar, 'x2')
    _, cpl = static(planar)
    ss = plate(model)
    for edge in ('x1', 'x2'):
        for field in ('u', 'v', 'w', 'phiy'):
            setattr(ss, edge + field, 0.)
    _, css = static(ss)
    w_face = center_w(s, c)
    w_planar = center_w(planar, cpl)
    w_ss = center_w(ss, css)
    assert w_planar < 0.99*w_face
    assert w_face < 0.99*w_ss
    # central deflections w D/(q a^4) of the documentation
    D = E*H**3/(12*(1 - NU**2))
    for w, ref in ((w_planar, 0.002901), (w_face, 0.002953), (w_ss, 0.013459)):
        assert np.isclose(w*D/(Q*A**4), ref, rtol=2e-4)
    ys = np.linspace(0, B, 11)
    for x in (0., A):
        f = fields(s, c, np.full_like(ys, x), ys)
        # the face displacement inside the plate
        scale = np.abs(uv_at(s, fields(s, c, np.full_like(ys, A/4), ys),
                             H/2, H)[0]).max()
        for z in (-H/2, H/2):
            u, v = uv_at(s, f, z, H)
            assert np.abs(u).max() < 1e-8*scale
            assert np.abs(v).max() < 1e-8*scale
        assert np.abs(f['w']).max() < 1e-8*w_face
        # warping: phi = w,x/2 and u(z) = (w,x/2) zeta (1 - 4 zeta^2/h^2)
        assert np.allclose(f['gxz'], 3*f['phix'], rtol=1e-6, atol=0)
        u, _ = uv_at(s, f, H/4, H)
        assert np.abs(u).max() > 1e3*np.abs(uv_at(s, f, H/2, H)[0]).max()
        assert np.allclose(u, f['gxz']/3*H/4*(1 - 4*(H/4)**2/H**2),
                           rtol=1e-6, atol=0)

In [14]:
test_face_clamp_tsdt_thick_plate()
print('test_face_clamp_tsdt_thick_plate: passed')

test_face_clamp_tsdt_thick_plate: passed


Both clamps are defined by the physical section, and the results do not
depend on the reference surface of the laminate:

In [15]:
# from tests/multidomain/test_face_clamp.py
def test_face_clamp_does_not_depend_on_the_reference_surface():
    model = 'plate_tsdt_donnell'
    xs, ys = np.meshgrid(np.linspace(0, A, 5), np.linspace(0, B, 5))
    res = {}
    for offset in (0., 0.3*H, 0.5*H):
        s = plate(model, offset)
        for x in (0., A):
            face_clamp(s, x, H)
        _, c = static(s)
        t = plate(model, offset)
        clamp_by_flags(t, 'x1')
        clamp_by_flags(t, 'x2')
        _, ct = static(t)
        res[offset] = (fields(s, c, xs.ravel(), ys.ravel())['w'],
                       fields(t, ct, xs.ravel(), ys.ravel())['w'])
    for offset in (0.3*H, 0.5*H):
        for i in (0, 1):
            assert np.allclose(res[offset][i], res[0.][i], rtol=1e-7, atol=0)
    assert res[0.][1].max() < 0.99*res[0.][0].max()

In [16]:
test_face_clamp_does_not_depend_on_the_reference_surface()
print('test_face_clamp_does_not_depend_on_the_reference_surface: passed')

test_face_clamp_does_not_depend_on_the_reference_surface: passed


With the penalty method, the kernels of the prescribed displacements are
given the same `z`:

In [17]:
# from tests/multidomain/test_face_clamp.py
def test_face_clamp_penalty_and_null_space_agree(offset):
    model = 'plate_tsdt_donnell'
    delta = 1e-5

    def build(k):
        s = plate(model, offset)
        face_clamp(s, 0., H, k=k)
        for z in (offset - H/2, offset + H/2):
            s.add_distr_pd_fixed_x(A, ku=k, kv=k, funcu=lambda y: -delta,
                                   funcv=lambda y: 0., z=z)
        s.add_distr_pd_fixed_x(A, kw=k, funcw=lambda y: 0.)
        s._rebuild()
        return s

    s = build(1.)
    _, c = static(s)
    k = 1e4*E
    p = build(k)
    md = MultiDomain([p], conn=[])
    size = md.get_size()
    kCp = 0
    for x in (0., A):
        for z in (offset - H/2, offset + H/2):
            kCp = kCp + fkCld_xcte(k, k, 0., p, x, size, 0, 0, z=z)
        kCp = kCp + fkCld_xcte(0., 0., k, p, x, size, 0, 0)
    K = md.calc_kC() + finalize_symmetric_matrix(kCp)
    cp = solve(K, md.calc_fext(), silent=True)
    xs, ys = np.meshgrid(np.linspace(0, A, 6), np.linspace(0, B, 5))
    f = fields(s, c, xs.ravel(), ys.ravel())
    fp = fields(p, cp, xs.ravel(), ys.ravel())
    for key in ('u', 'w'):
        assert np.allclose(fp[key], f[key], rtol=0,
                           atol=1e-3*np.abs(f[key]).max())
    # the faces of x = a are at -delta
    f = fields(s, c, np.full(5, A), np.linspace(0, B, 5))
    for z in (offset - H/2, offset + H/2):
        assert np.allclose(uv_at(s, f, z, H)[0], -delta, rtol=1e-8)

In [18]:
for offset in [0.0, 0.3 * H]:
    test_face_clamp_penalty_and_null_space_agree(offset=offset)
    print(f"test_face_clamp_penalty_and_null_space_agree(offset={offset!r}): passed")

test_face_clamp_penalty_and_null_space_agree(offset=0.0): passed


test_face_clamp_penalty_and_null_space_agree(offset=0.015): passed
